## Housekeeping

### Import dependencies and silence warnings

In [ ]:
import polars as pl
from pathlib import Path

from process_data import process_pbp_metadata, aggregate_stints_df
from regressions import build_position_map, build_matrix, run_all_regressions

import chickenstats.utilities

import matplotlib.pyplot as plt

import seaborn as sns

### Polars configuration options

In [ ]:
pl.Config.set_tbl_cols(-1)

polars.config.Config

### Setting up the chickenstats styles

In [ ]:
plt.style.use("chickenstats")

## Illustrative data munging and matrix constructions

### Reading play-by-play data as a lazyframe

In [ ]:
pbp_path = Path("./data/raw/pbp.parquet")

lazy_pbp = pl.scan_parquet(pbp_path)

### Choosing the season(s) for this run

In [ ]:
# Settings the season as the latest full season in the data
seasons = [20242025]

# just so you can see them all
sorted(lazy_pbp.select(pl.col("season")).unique().collect()["season"].to_list())

[20102011,
 20112012,
 20122013,
 20132014,
 20142015,
 20152016,
 20162017,
 20172018,
 20182019,
 20192020,
 20202021,
 20212022,
 20222023,
 20232024,
 20242025]

### Filtering pbp to a single season to show data processing

In [ ]:
# Filtering pbp data to a single season / session combo to illustrate
# what goes in in the background prior to running the regression
conditions = (pl.col("season") == seasons[0], pl.col("session") == "R")
season_pbp = lazy_pbp.filter(conditions).collect()

### Creating the lookup table for back-to-back games

In [ ]:
# Lookup table for back-to-back games, leverages same conditions as the previous cell
meta_lookup = process_pbp_metadata(lazy_pbp.filter(conditions))

### Aggregating to stints-level dataframe

In [ ]:
# Aggregating to the stints-level dataframe, using the season data and the b2b meta lookup table
stints = aggregate_stints_df(pbp_df=season_pbp, meta_lookup=meta_lookup)

In [ ]:
# Showing the stints in order for 2024-25 season
stints.sort(["game_id", "period", "stint_id"], descending=[False, False, False])

season,session,game_id,period,stint_id,toi,h_xgf,a_xgf,h_cf,a_cf,h_gf,a_gf,h_ids,a_ids,h_goalie_ids,a_goalie_ids,h_team,a_team,h_s3,h_s7,a_s3,a_s7,h_b2b,a_b2b,strength,ozs,nzs,dzs,h_skaters,a_skaters,h_goalies,a_goalies,h_cnt,a_cnt
i64,str,i64,i64,i32,i64,f64,f64,u32,u32,u32,u32,list[str],list[str],list[str],list[str],str,str,i32,i64,i32,i64,bool,bool,str,bool,bool,bool,list[str],list[str],list[str],list[str],u32,u32
20242025,"""R""",2024020001,1,6,21,0.0,0.012213,0,1,0,0,"[""HENRI.JOKIHARJU"", ""RASMUS.DAHLIN"", … ""BECK.MALENSTYN""]","[""JONAS.SIEGENTHALER"", ""SIMON.NEMEC"", … ""NICO.HISCHIER""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]","""BUF""","""NJD""",0,0,0,0,false,false,"""5v5""",false,true,false,"[""HENRI.JOKIHARJU"", ""RASMUS.DAHLIN"", … ""SAM.LAFFERTY""]","[""JONAS.SIEGENTHALER"", ""SIMON.NEMEC"", … ""ONDREJ.PALAT""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]",5,5
20242025,"""R""",2024020001,1,8,1,0.0,0.0,0,0,0,0,"[""HENRI.JOKIHARJU"", ""BOWEN.BYRAM"", … ""JJ.PETERKA""]","[""JONAS.SIEGENTHALER"", ""SIMON.NEMEC"", … ""NICO.HISCHIER""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]","""BUF""","""NJD""",0,0,0,0,false,false,"""5v5""",false,false,false,"[""HENRI.JOKIHARJU"", ""BOWEN.BYRAM"", … ""SAM.LAFFERTY""]","[""JONAS.SIEGENTHALER"", ""SIMON.NEMEC"", … ""TIMO.MEIER""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]",5,5
20242025,"""R""",2024020001,1,9,1,0.0,0.0,0,0,0,0,"[""HENRI.JOKIHARJU"", ""BOWEN.BYRAM"", … ""JJ.PETERKA""]","[""JONAS.SIEGENTHALER"", ""SIMON.NEMEC"", … ""JESPER.BRATT""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]","""BUF""","""NJD""",0,0,0,0,false,false,"""5v5""",false,false,false,"[""HENRI.JOKIHARJU"", ""BOWEN.BYRAM"", … ""SAM.LAFFERTY""]","[""JONAS.SIEGENTHALER"", ""SIMON.NEMEC"", … ""TIMO.MEIER""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]",5,5
20242025,"""R""",2024020001,1,11,6,0.0,0.019322,0,1,0,0,"[""HENRI.JOKIHARJU"", ""BOWEN.BYRAM"", … ""JJ.PETERKA""]","[""JONAS.SIEGENTHALER"", ""SIMON.NEMEC"", … ""JACK.HUGHES""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]","""BUF""","""NJD""",0,0,0,0,false,false,"""5v5""",false,false,false,"[""HENRI.JOKIHARJU"", ""BOWEN.BYRAM"", … ""SAM.LAFFERTY""]","[""JONAS.SIEGENTHALER"", ""SIMON.NEMEC"", … ""JESPER.BRATT""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]",5,5
20242025,"""R""",2024020001,1,12,4,0.0,0.0,0,0,0,0,"[""HENRI.JOKIHARJU"", ""BOWEN.BYRAM"", … ""JJ.PETERKA""]","[""DOUGIE.HAMILTON"", ""SIMON.NEMEC"", … ""JACK.HUGHES""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]","""BUF""","""NJD""",0,0,0,0,false,false,"""5v5""",false,false,false,"[""HENRI.JOKIHARJU"", ""BOWEN.BYRAM"", … ""SAM.LAFFERTY""]","[""DOUGIE.HAMILTON"", ""SIMON.NEMEC"", … ""JESPER.BRATT""]","[""UKKO-PEKKA.LUUKKONEN""]","[""JACOB.MARKSTROM""]",5,5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
20242025,"""R""",2024021312,3,144,1,0.0,0.022554,0,1,0,0,"[""IVAN.PROVOROV"", ""DENTON.MATEYCHUK"", … ""ADAM.FANTILLI""]","[""SCOTT.MAYFIELD"", ""TONY.DEANGELO"", … ""MARC.GATCOMB""]","[""JET.GREAVES""]","[""MARCUS.HOGBERG""]","""CBJ""","""NYI""",1,3,-1,-3,false,false,"""5v5""",false,false,false,"[""IVAN.PROVOROV"", ""DENTON.MATEYCHUK"", … ""KENT.JOHNSON""]","[""SCOTT.MAYFIELD"", ""TONY.DEANGELO"", … ""KYLE.MACLEAN""]","[""JET.GREAVES""]","[""MARCUS.HOGBERG""]",5,5
20242025,"""R""",2024021312,3,146,1,0.0,0.0,0,0,0,0,"[""IVAN.PROVOROV"", ""DENTON.MATEYCHUK"", … ""ADAM.FANTILLI""]","[""SCOTT.MAYFIELD"", ""TONY.DEANGELO"", … ""KYLE.MACLEAN""]","[""JET.GREAVES""]","[""MARCUS.HOGBERG""]","""CBJ""","""NYI""",1,3,-1,-3,false,false,"""5v5""",false,false,false,"[""IVAN.PROVOROV"", ""DENTON.MATEYCHUK"", … ""KIRILL.MARCHENKO""]","[""SCOTT.MAYFIELD"", ""TONY.DEANGELO"", … ""CASEY.CIZIKAS""]","[""JET.GREAVES""]","[""MARCUS.HOGBERG""]",5,5
20242025,"""R""",2024021312,3,147,2,0.0,0.0,0,0,0,0,"[""IVAN.PROVOROV"", ""DENTON.MATEYCHUK"", … ""ADAM.FANTILLI""]","[""SCOTT.MAYFIELD"", ""SCOTT.PERUNOVICH"", … ""HUDSON.FASCHING"

### Building the regression matrix, target / weights numpy arrays, full skater list, and the player metrics for that season

In [ ]:
# Function returns a bunch of different data for us
X, Y, W, skater_list, player_metrics = build_matrix(stints)

### Sparse matrix for running the regressions

In [ ]:
# The sparse matrix used in the ridge regression only stores the 1 values,
# with the x coordinates indicating the index of the stint and the y-coordinates
# indicating the players who are on-ice offensive / defensively
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 11401669 stored elements and shape (923652, 2067)>
  Coords	Values
  (0, 128)	1.0
  (0, 240)	1.0
  (0, 437)	1.0
  (0, 467)	1.0
  (0, 782)	1.0
  (0, 1483)	1.0
  (0, 1502)	1.0
  (0, 1612)	1.0
  (0, 1652)	1.0
  (0, 1716)	1.0
  (0, 2056)	1.0
  (0, 2063)	1.0
  (0, 2066)	1.0
  (1, 169)	1.0
  (1, 457)	1.0
  (1, 528)	1.0
  (1, 774)	1.0
  (1, 860)	1.0
  (1, 1029)	1.0
  (1, 1342)	1.0
  (1, 1419)	1.0
  (1, 1566)	1.0
  (1, 1828)	1.0
  (1, 2056)	1.0
  (1, 2063)	1.0
  :	:
  (923649, 2056)	1.0
  (923649, 2065)	1.0
  (923650, 88)	1.0
  (923650, 207)	1.0
  (923650, 299)	1.0
  (923650, 479)	1.0
  (923650, 747)	1.0
  (923650, 1463)	1.0
  (923650, 1613)	1.0
  (923650, 1721)	1.0
  (923650, 1744)	1.0
  (923650, 1954)	1.0
  (923650, 2056)	1.0
  (923651, 33)	1.0
  (923651, 449)	1.0
  (923651, 708)	1.0
  (923651, 791)	1.0
  (923651, 864)	1.0
  (923651, 1072)	1.0
  (923651, 1428)	1.0
  (923651, 1583)	1.0
  (923651, 1822)	1.0
  (923651, 1970)	1.0
  (9

### Numpy array of the target metric, in this case xG / 60

In [ ]:
# Array of stint-level xG / 60, used in the ridge regression
print(Y)

[0. 0. 0. ... 0. 0. 0.]


### Numpy array of the weights, which are time-on-ice

In [ ]:
# Array of the stint-level time-on-ice
print(W)

[ 1  2  9 ... 10  3  1]


### List of skaters that season who had more than the time-on-ice minimum (in this case, one minute)

In [ ]:
# All of the skaters who appear on the ice for more than one minute, aggregated by team
skater_list

['A.J.GREER_FLA',
 'AARON.EKBLAD_FLA',
 'AATU.RATY_VAN',
 'ADAM.BOQVIST_FLA',
 'ADAM.BOQVIST_NYI',
 'ADAM.EDSTROM_NYR',
 'ADAM.FANTILLI_CBJ',
 'ADAM.FOX_NYR',
 'ADAM.GAUDETTE_OTT',
 'ADAM.GINNING_PHI',
 'ADAM.HENRIQUE_EDM',
 'ADAM.KLAPKA_CGY',
 'ADAM.LARSSON_SEA',
 'ADAM.LOWRY_WPG',
 'ADAM.PELECH_NYI',
 'ADAM.WILSBY_NSH',
 'ADRIAN.KEMPE_LAK',
 'AKIL.THOMAS_LAK',
 'ALBERT.JOHANSSON_DET',
 'ALEC.MARTINEZ_CHI',
 'ALEKSANDER.BARKOV_FLA',
 'ALEX.BARRE-BOULET_MTL',
 'ALEX.DEBRINCAT_DET',
 'ALEX.IAFALLO_WPG',
 'ALEX.KILLORN_ANA',
 'ALEX.LAFERRIERE_LAK',
 'ALEX.NEWHOOK_MTL',
 'ALEX.NYLANDER_TOR',
 'ALEX.OVECHKIN_WSH',
 'ALEX.PIETRANGELO_VGK',
 'ALEX.STEEVES_TOR',
 'ALEX.TUCH_BUF',
 'ALEX.TURCOTTE_LAK',
 'ALEX.VLASIC_CHI',
 'ALEXANDER.ALEXEYEV_WSH',
 'ALEXANDER.HOLTZ_VGK',
 'ALEXANDER.KERFOOT_UTA',
 'ALEXANDER.PETROVIC_DAL',
 'ALEXANDER.ROMANOV_NYI',
 'ALEXANDER.WENNBERG_SJS',
 'ALEXANDRE.CARRIER_MTL',
 'ALEXANDRE.CARRIER_NSH',
 'ALEXANDRE.TEXIER_STL',
 'ALEXEI.TOROPCHENKO_STL',
 'ALEXIS.LAFREN

### Skater metrics for the given time period, in this case EV xG in 2024-2025 season

In [ ]:
player_metrics

player_team,toi,metric_for,metric_against
str,i64,f64,f64
"""NIKITA.KUCHEROV_TBL""",77688,67.342093,57.792191
"""JONAS.RONDBJERG_VGK""",7204,3.728668,4.519087
"""FABIAN.ZETTERLUND_OTT""",14848,10.9658,11.126272
"""SAM.BENNETT_FLA""",65631,58.097245,49.081703
"""JACOB.GAUCHER_PHI""",1762,1.444801,2.209146
…,…,…,…
"""PIERRE-LUC.DUBOIS_WSH""",71442,61.654174,48.852618
"""MARTIN.NECAS_COL""",29200,29.857786,19.199912
"""SAM.LAFFERTY_BUF""",33545,16.496754,23.334136


## Illustrative regression run

### Build the map of players and positions (for use in the regression training runs)

In [ ]:
position_map = build_position_map(lazy_pbp)

### Setting the parameters for the run

In [ ]:
# Time-on-ice limits for players to consider in the regression
toi_limits = dict(ev_r=1, ev_p=1, other_r=1, other_p=1)

# Metrics to loop through
metrics = ["xg", "corsi", "goals"]

# Sessions (i.e., regular season, playoffs) to loop through
sessions = ["R", "P"]

# Situations to loop through
situations = ["EV", "PP", "SH"]

### Setting the directory where the stints files are saved

In [ ]:
# Directory where the aggregated stints dataframes are stored
stints_directory = Path("./data/processed/")

### Running the regression

In [ ]:
# Running the regression for the years selected, should take 60-90 seconds per year
regression_results = run_all_regressions(
    stints_directory=stints_directory,
    position_map=position_map,
    seasons=seasons,
    sessions=sessions,
    situations=situations,
    metrics=metrics,
    toi_limits=toi_limits,
)

Output()

In [ ]:
# Filtering dataframe just to show a piece of the results
filter_conditions = (pl.col("session") == "R", pl.col("season") == 20242025, pl.col("situation") == "EV")

# Top players by offensive xG z-score in 2024-25 season
regression_results.filter(filter_conditions).sort("off_coeff_xg_z", descending=True).head(10)

season,session,player,team,pos,situation,toi_minutes,off_coeff_xg,off_coeff_corsi,off_coeff_goals,def_coeff_xg,def_coeff_corsi,def_coeff_goals,metric_for_xg,metric_for_corsi,metric_for_goals,metric_against_xg,metric_against_corsi,metric_against_goals,metric_diff_xg,metric_diff_corsi,metric_diff_goals,on_ice_for_60_xg,on_ice_for_60_corsi,on_ice_for_60_goals,on_ice_against_60_xg,on_ice_against_60_corsi,on_ice_against_60_goals,on_ice_diff_60_xg,on_ice_diff_60_corsi,on_ice_diff_60_goals,total_rapm_xg,total_rapm_corsi,total_rapm_goals,off_coeff_xg_z,off_coeff_corsi_z,off_coeff_goals_z,def_coeff_xg_z,def_coeff_corsi_z,def_coeff_goals_z,metric_for_xg_z,metric_for_corsi_z,metric_for_goals_z,metric_against_xg_z,metric_against_corsi_z,metric_against_goals_z,metric_diff_xg_z,metric_diff_corsi_z,metric_diff_goals_z,on_ice_for_60_xg_z,on_ice_for_60_corsi_z,on_ice_for_60_goals_z,on_ice_against_60_xg_z,on_ice_against_60_corsi_z,on_ice_against_60_goals_z,on_ice_diff_60_xg_z,on_ice_diff_60_corsi_z,on_ice_diff_60_goals_z,total_rapm_xg_z,total_rapm_corsi_z,total_rapm_goals_z
i32,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20242025,"""R""","""LEON.DRAISAITL""","""EDM""","""C""","""EV""",1226.033333,0.584323,3.243185,0.460713,0.037443,-0.717157,-0.109602,78.289765,1370.0,79.0,50.350188,1132.0,47.0,27.939577,238.0,32.0,3.831369,67.045485,3.866127,2.464053,55.398168,2.300101,1.367316,11.647318,1.566026,0.54688,3.960342,0.570315,4.123967,2.316678,3.780826,0.223929,-0.550304,-1.411486,2.636247,1.806504,2.518587,1.285685,1.26313,1.157064,4.624816,3.79521,3.303906,2.397194,1.457444,1.551808,-0.142706,-0.385944,-0.1809,1.526398,1.269908,0.997946,3.236739,2.161709,3.924842
20242025,"""R""","""NATHAN.MACKINNON""","""COL""","""C""","""EV""",1395.616667,0.573862,5.242333,0.150314,0.044993,-0.53728,-0.079801,82.920234,1563.0,73.0,61.644183,1345.0,51.0,21.27605,218.0,22.0,3.564886,67.196102,3.138398,2.650191,57.823901,2.192579,0.914695,9.372201,0.945818,0.528869,5.779613,0.230115,4.049216,3.724405,1.196441,0.29691,-0.41195,-1.052371,2.869655,2.259382,2.236441,1.888514,1.770763,1.372595,3.532459,3.478761,2.274184,1.960319,1.479008,0.857176,0.103079,-0.02635,-0.254415,1.086433,1.043888,0.670525,3.131508,3.14356,1.587376
20242025,"""R""","""BRADY.TKACHUK""","""OTT""","""L""","""EV""",1014.783333,0.46874,5.328876,0.075095,-0.103767,0.193725,-0.081022,53.036426,1143.0,43.0,38.969094,966.0,38.0,14.067332,177.0,5.0,3.135828,67.580929,2.542415,2.304084,57.11564,2.246785,0.831744,10.465288,0.29563,0.572507,5.13515,0.156118,3.952793,3.998876,0.520782,-1.047927,0.239869,-1.058541,1.5374,1.450137,0.910524,0.875617,1.048422,0.866965,2.454352,2.710472,0.443884,1.096771,1.441103,0.276754,-0.238695,-0.011987,-0.278665,1.127385,0.999005,0.367644,3.520702,2.889769,1.005561
20242025,"""R""","""ZACH.WERENSKI""","""CBJ""","""D""","""EV""",1765.866667,0.467824,6.016223,0.150246,0.006099,0.854679,-0.121615,92.467339,1959.0,91.0,79.637238,1793.0,68.0,12.830101,166.0,23.0,3.141823,66.562217,3.091966,2.705886,60.921927,2.31048,0.435937,5.64029,0.781486,0.461725,5.161544,0.271861,3.802575,4.016401,1.550142,-0.00321,0.67741,-1.330779,2.616944,2.378699,2.56105,2.178148,2.111892,1.694212,2.015398,2.332638,2.438694,1.16413,1.343846,0.829539,0.383811,0.510029,-0.192526,0.640352,0.631065,0.614183,2.838515,2.677714,2.084978
20242025,"""R""","""CONNOR.MCDAVID""","""EDM""","""C""","""EV""",1193.6,0.527546,2.165932,0.445872,-0.016668,-1.557722,0.122605,77.902676,1305.0,80.0,47.969066,1078.0,58.0,29.93361,227.0,22.0,3.916019,65.599866,4.021448,2.411314,54.189008,2.91555,1.504706,11.410858,1.105898,0.544214,3.723655,0.323267,3.718271,1.558116,3.657266,-0.299162,-1.196837,1.386701,2.616735,1.65398,2.565612,1.158591,1.134435,1.749772,4.951699,3.621163,2.274184,2.53597,1.250466,1.700065,-0.212346,-0.565192,0.

## Analyzing the RAPM model

### Load data

In [ ]:
parquet_file = Path("./data/results/results.parquet")

rapm_results = pl.read_parquet(parquet_file)

### Previewing the results

In [ ]:
# Filtering dataframe just to show a piece of the results
filter_conditions = (pl.col("session") == "R", pl.col("situation") == "EV")

# Top regular season, even strength defensive players, all-time
rapm_results.filter(filter_conditions).sort(pl.col("def_coeff_xg_z"), descending=False).head(10)

season,session,player,team,pos,situation,toi_minutes,off_coeff_goals,off_coeff_xg,off_coeff_corsi,def_coeff_goals,def_coeff_xg,def_coeff_corsi,metric_for_goals,metric_for_xg,metric_for_corsi,metric_against_goals,metric_against_xg,metric_against_corsi,metric_diff_goals,metric_diff_xg,metric_diff_corsi,on_ice_for_60_goals,on_ice_for_60_xg,on_ice_for_60_corsi,on_ice_against_60_goals,on_ice_against_60_xg,on_ice_against_60_corsi,on_ice_diff_60_goals,on_ice_diff_60_xg,on_ice_diff_60_corsi,total_rapm_xg,total_rapm_corsi,total_rapm_goals,off_coeff_goals_z,off_coeff_xg_z,off_coeff_corsi_z,def_coeff_goals_z,def_coeff_xg_z,def_coeff_corsi_z,metric_for_goals_z,metric_for_xg_z,metric_for_corsi_z,metric_against_goals_z,metric_against_xg_z,metric_against_corsi_z,metric_diff_goals_z,metric_diff_xg_z,metric_diff_corsi_z,on_ice_for_60_goals_z,on_ice_for_60_xg_z,on_ice_for_60_corsi_z,on_ice_against_60_goals_z,on_ice_against_60_xg_z,on_ice_against_60_corsi_z,on_ice_diff_60_goals_z,on_ice_diff_60_xg_z,on_ice_diff_60_corsi_z,total_rapm_xg_z,total_rapm_corsi_z,total_rapm_goals_z
i32,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20202021,"""R""","""CHRIS.TANEV""","""CGY""","""D""","""EV""",1007.016667,-0.059216,0.072002,0.083065,-0.366596,-0.363259,-5.579549,39.0,41.172539,898.0,26.0,27.712755,731.0,13.0,13.459784,167.0,2.323695,2.453139,53.504576,1.54913,1.65118,43.554393,0.774565,0.80196,9.950183,0.435261,5.662614,0.30738,-0.523107,0.820493,0.122873,-3.360254,-3.743327,-4.559412,1.112617,1.484479,1.385963,0.353766,0.581432,0.885458,1.617704,2.990158,3.545124,0.028078,0.424792,0.24089,-0.668762,-1.187375,-1.451515,0.605142,1.036631,1.22873,3.374994,3.653765,1.873677
20152016,"""R""","""PAVEL.DATSYUK""","""DET""","""C""","""EV""",1068.283333,0.050715,0.380992,2.771684,-0.135756,-0.397038,-4.129011,39.0,48.644313,966.0,32.0,33.711129,850.0,7.0,14.933184,116.0,2.19043,2.732102,54.255269,1.797276,1.893381,47.740144,0.393154,0.83872,6.515126,0.77803,6.900695,0.186472,0.361331,2.479933,1.901452,-1.626313,-3.692033,-3.534543,0.848508,1.413268,1.046928,0.579012,0.623875,0.780269,0.762677,3.104795,1.995322,0.108166,0.971936,0.302607,-0.294424,-0.703785,-0.847993,0.244233,1.245792,0.913099,4.372971,4.014182,1.323148
20172018,"""R""","""MIKKO.KOIVU""","""MIN""","""C""","""EV""",1105.533333,-0.154724,-0.092337,-1.700992,-0.146231,-0.325874,-4.729377,40.0,42.913629,1002.0,38.0,30.061758,919.0,2.0,12.851871,83.0,2.170898,2.329028,54.380993,2.062353,1.631525,49.876379,0.108545,0.697502,4.504613,0.233537,3.028384,-0.008492,-1.033155,-0.841742,-1.245865,-1.302721,-3.575285,-3.768489,0.699172,0.936731,0.982166,0.696726,0.279994,0.802726,0.186511,2.711927,1.396161,-0.021222,0.12191,-0.142069,-0.293831,-1.576979,-1.106719,0.165184,1.264204,0.631178,1.704254,1.767516,-0.068398
20192020,"""R""","""ZACH.ASTON-REESE""","""PIT""","""C""","""EV""",688.883333,-0.085907,0.011319,-0.463139,-0.195787,-0.304132,-1.947085,22.0,23.154253,575.0,17.0,17.226876,545.0,5.0,5.927377,30.0,1.916144,2.016677,50.081049,1.480657,1.500417,47.468125,0.435487,0.51626,2.612924,0.315451,1.483946,0.10988,-0.794391,-0.006208,-0.407306,-2.16836,-3.554963,-2.137178,0.023584,0.157891,0.285848,-0.285191,-0.232835,0.197634,0.583434,1.492615,0.699155,-0.285886,-0.452531,-0.673373,-0.755559,-1.729409,-1.273155,0.374157,0.861476,0.441292,2.426574,1.209034,0.785079
20142015,"""R""","""JUSTIN.ABDELKADER""","""DET""","""L""","""EV""",977.766667,0.060134,0.047285,-1.563983,-0.062738,-0.313783,-3.048697,40.0,37.027031,780.0,31.0,27.301037,752.0,9.0,9.725994,28.0,2.454573,2.272139,47.86418,1.902294,1.67531,46.145979,0.552279,0.596829,1.718201,0.361068,1.484713,0.122872,0.743283,0.389809,-1.1055,-1.216323,-3.5115,-2.529366,1.103063,1.050761,0.802148,0.683656,0.458302,0.731429,1.117291,1.953751,0.535765,0.51302,0.288345,-0.6

In [ ]:
# Followed by top regular season, even strength offensive players, all-time
rapm_results.filter(filter_conditions).sort(pl.col("off_coeff_xg_z"), descending=True).head(10)

season,session,player,team,pos,situation,toi_minutes,off_coeff_goals,off_coeff_xg,off_coeff_corsi,def_coeff_goals,def_coeff_xg,def_coeff_corsi,metric_for_goals,metric_for_xg,metric_for_corsi,metric_against_goals,metric_against_xg,metric_against_corsi,metric_diff_goals,metric_diff_xg,metric_diff_corsi,on_ice_for_60_goals,on_ice_for_60_xg,on_ice_for_60_corsi,on_ice_against_60_goals,on_ice_against_60_xg,on_ice_against_60_corsi,on_ice_diff_60_goals,on_ice_diff_60_xg,on_ice_diff_60_corsi,total_rapm_xg,total_rapm_corsi,total_rapm_goals,off_coeff_goals_z,off_coeff_xg_z,off_coeff_corsi_z,def_coeff_goals_z,def_coeff_xg_z,def_coeff_corsi_z,metric_for_goals_z,metric_for_xg_z,metric_for_corsi_z,metric_against_goals_z,metric_against_xg_z,metric_against_corsi_z,metric_diff_goals_z,metric_diff_xg_z,metric_diff_corsi_z,on_ice_for_60_goals_z,on_ice_for_60_xg_z,on_ice_for_60_corsi_z,on_ice_against_60_goals_z,on_ice_against_60_xg_z,on_ice_against_60_corsi_z,on_ice_diff_60_goals_z,on_ice_diff_60_xg_z,on_ice_diff_60_corsi_z,total_rapm_xg_z,total_rapm_corsi_z,total_rapm_goals_z
i32,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20212022,"""R""","""CONNOR.MCDAVID""","""EDM""","""C""","""EV""",1394.466667,0.397633,0.673987,5.472745,-0.118531,-0.011333,-0.700795,80.0,85.73567,1493.0,54.0,61.807032,1318.0,26.0,23.928638,175.0,3.442176,3.688966,64.239614,2.323469,2.659384,56.709853,1.118707,1.029582,7.52976,0.68532,6.17354,0.516164,2.570962,5.237508,4.069536,-1.095333,-0.211794,-0.515084,2.546571,3.262885,2.511506,1.467876,2.138798,2.110723,2.866677,3.9764,2.562492,0.734059,2.2053,1.42545,-0.2781,0.229795,0.302317,0.890953,1.455595,0.98137,4.478564,3.439987,3.119434
20222023,"""R""","""MATTHEW.TKACHUK""","""FLA""","""L""","""EV""",1165.65,0.628156,0.893519,6.788146,0.059458,0.041857,-0.3477,92.0,82.705709,1360.0,54.0,56.536985,1147.0,38.0,26.168724,213.0,4.735555,4.257146,70.003861,2.779565,2.910152,59.040021,1.95599,1.346994,10.96384,0.851662,7.135845,0.568698,5.553191,4.996691,4.849043,0.629972,0.261431,-0.31884,3.36201,2.967108,2.180759,1.721853,1.8767,1.729021,4.135263,4.147268,3.442465,2.294521,1.974357,1.607432,0.090745,0.31631,0.373133,1.332998,1.241299,1.180201,3.999618,4.443887,4.387392
20172018,"""R""","""CONNOR.MCDAVID""","""EDM""","""C""","""EV""",1400.233333,0.629181,0.617871,4.504192,0.014989,0.248943,1.242842,87.0,82.456034,1555.0,64.0,69.52803,1396.0,23.0,12.928003,159.0,3.72795,3.533241,66.631752,2.7424,2.979276,59.818602,0.98555,0.553965,6.81315,0.368928,3.26135,0.614192,3.875569,4.884521,3.229498,0.081201,2.541251,1.006721,2.907438,2.997698,2.293641,2.088266,2.453172,1.957303,2.288781,2.727947,2.682229,1.226241,2.468641,1.549941,0.384238,1.194713,0.435194,0.705331,1.045411,0.869755,2.700551,1.905429,3.150795
20112012,"""R""","""JOHN.TAVARES""","""NYI""","""C""","""EV""",1382.083333,0.220986,0.55102,2.906426,0.134463,0.079714,1.140196,66.0,75.426779,1378.0,66.0,61.566216,1319.0,0.0,13.860563,59.0,2.86524,3.274482,59.822731,2.86524,2.672757,57.261381,0.0,0.601725,2.561351,0.471305,1.76623,0.086523,1.941257,4.531727,2.172841,1.604662,0.869882,0.937496,2.256468,3.04401,2.190267,2.572762,2.424081,2.088968,-0.021564,2.755739,0.981788,0.874518,2.313028,0.987345,0.71729,0.902748,0.518893,0.160831,1.054394,0.311235,3.208636,1.021242,0.642443
20202021,"""R""","""CONNOR.MCDAVID""","""EDM""","""C""","""EV""",959.45,0.541102,0.480818,4.205118,0.224712,0.029972,0.217301,66.0,52.852726,975.0,51.0,40.847974,910.0,15.0,12.004751,65.0,4.127365,3.305189,60.972432,3.189327,2.554462,56.907603,0.938037,0.750727,4.064829,0.450846,3.987817,0.31639,4.236425,4.410779,3.572983,2.050196,0.286616,0.20807,3.543236,3.130503,2.450048,2.838341,2.304327,2.244426,1.978145,2.979405,1.547153,1.578622,2.187041,1.423837,0.562489,0.410278,0.574685,0.613364,1.219819,0.617332,3.293293,2